# Satellite validation and calibration of the offshore hindcast

Compares the offshore hindcast at the goals (`offshore_waves`, e.g. WHACS) with
satellite altimetry, and calibrates it per goal:

1. **Passes at each goal** — altimeter observations within `radius_km` of each
   goal's hindcast node, in water deeper than `min_depth`, with good quality
   flags; one median Hs per satellite pass; the hindcast at the pass time.
2. **Calibration per goal** — directional calibration (wind sea and swell, per
   direction sector; Mínguez et al., 2011) fitted on `train_fraction` of the
   passes (75 % by default) and validated on the rest.
3. **Calibrated hindcast** — written as a new zarr, to add to `offshore_waves`
   (e.g. `WHACS_cal`) and use in 08 onwards like any other offshore source.

Everything comes from the `satellite` block of the case JSON. Nothing here is
specific to one hindcast: any `offshore_waves` entry on `(time, goal)` with
`hs`, `dir` (and partitions `phs{i}`, `pdir{i}`) works.

**Prerequisites:** `07a_Offshore_WHACS.ipynb` or `07a_Offshore_ERA5.ipynb` (the goal
zarr) and the altimeter store (`satellite.data_path`). Method:
`bluemath_tk.waves.calibration` and `bluemath_tk.waves.altimetry`.

In [ ]:
import json
import os
import sys

# GDAL/PROJ data paths when the kernel was not started through `conda activate`
os.environ.setdefault("GDAL_DATA", os.path.join(sys.prefix, "share", "gdal"))
os.environ.setdefault("PROJ_DATA", os.path.join(sys.prefix, "share", "proj"))

from utils import calibration as cal
from utils import viz

## Configuration

In [ ]:
offshore_name = "WHACS"  # offshore_waves entry to validate and calibrate (e.g. "ERA5")

cfg = cal.config()
print(json.dumps(cfg, indent=1))

## 1. Satellite passes at each goal

Cached in `outputs/calibration/{offshore_name}/pairs.csv`; `refresh=True`
recomputes them (e.g. after changing `radius_km` or `min_depth`).

In [ ]:
pairs = cal.satellite_pairs(offshore_name, refresh=False)
pairs.groupby("goal").size().rename("passes").to_frame().T

In [ ]:
fig = viz.plot_satellite_pairs(pairs, cal.goal_nodes(offshore_name), cfg["radius_km"])

## 2. Calibration per goal

Fitted on `train_fraction` of each goal's passes; the metrics below are on the
**test** passes, which the fit never saw (bias = hindcast − satellite).

In [ ]:
calibrations = cal.fit_calibration(offshore_name)
metrics = cal.calibration_metrics(offshore_name)
metrics[metrics["set"] == "test"].pivot(
    index="goal", columns="stage", values=["n", "bias", "rmse", "slope"]
).round(3)

In [ ]:
fig = viz.plot_calibration_metrics(metrics, subset="test")

Per goal: test passes before and after the calibration, and the Hs factor of
each direction sector for wind sea and swell (red above 1, blue below, grey =
not enough data, left at 1).

In [ ]:
for gid, calval in calibrations.items():
    calval.plot(title=f"{offshore_name} · goal {gid}")

## 3. Write the calibrated hindcast

Applies each goal's factors to the whole record (partition Hs times its
sector factor, bulk Hs rescaled; periods and directions unchanged) and writes
`*_cal.zarr` next to the source. Add the printed entry to `offshore_waves` in
the case JSON.

In [ ]:
entry = cal.write_calibrated_hindcast(offshore_name)
print(json.dumps(entry, indent=4))

## Next

- Add the entry above to `offshore_waves` in the case JSON.
- `08_Dynamic_Cases_Generation.ipynb` with `offshore_name = "WHACS_cal"` (the
  dynamic runs must be forced with the calibrated hindcast too, so that 09
  compares like with like), then 09–12 with the same name.
- `10_Validate_Buoys.ipynb` with both names: the buoys are an independent check
  of the calibration.